# Feature extraction
## Load Data

In [36]:
import os 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pvlib

In [40]:
base_dir = os.getcwd()
data_dir = os.path.join(base_dir, 'data')

ausgrid_ppv_filepath = os.path.join(data_dir, 'ausgrid_ppv.csv')
ausgrid_complete_filepath = os.path.join(data_dir, 'ausgrid_complete.csv')

ausgrid_ppv = pd.read_csv(ausgrid_ppv_filepath)
ausgrid_complete = pd.read_csv(ausgrid_complete_filepath)

# Convert datetimes
ausgrid_ppv["datetime"] = pd.to_datetime(ausgrid_ppv["datetime"])
ausgrid_complete["datetime"] = pd.to_datetime(ausgrid_complete["datetime"])

# Inspect data
print(f"Name: Ausgrid PPV Data")
print(f"Shape: {ausgrid_ppv.shape}")
print(f"Columns:", ausgrid_ppv.columns.tolist())
print(f"Missing values: {ausgrid_ppv.isna().sum().sum()}\n")

print(f"Name: Ausgrid Complete Data")
print(f"Shape: {ausgrid_complete.shape}")
print(f"Columns:", ausgrid_complete.columns.tolist())
print(f"Missing values: {ausgrid_complete.isna().sum().sum()}")

Name: Ausgrid PPV Data
Shape: (15778212, 13)
Columns: ['Customer', 'Generator Capacity', 'Postcode', 'datetime', 'PPV', 'temperature_2m', 'wind_speed_10m', 'shortwave_radiation', 'cloud_cover', 'direct_normal_irradiance', 'diffuse_radiation', 'sunshine_duration', 'sun_position']
Missing values: 0

Name: Ausgrid Complete Data
Shape: (7255446, 15)
Columns: ['Customer', 'Generator Capacity', 'Postcode', 'datetime', 'Pload', 'PPV', 'Pnet', 'temperature_2m', 'wind_speed_10m', 'shortwave_radiation', 'cloud_cover', 'direct_normal_irradiance', 'diffuse_radiation', 'sunshine_duration', 'sun_position']
Missing values: 0


## Weather Features
### Calculate Sun Position
$\theta_\text{sun} = \sin \phi \sin \delta + \cos \phi \cos \delta \cos H$
- $\delta$ = solar declination, determined by the date.
- $H$ = solar hour angle, determined by time of day.

$\delta = 23.45 \degree \sin (\frac{360 \degree}{365}(284 + n))$
- $n$ = day of year (1-365)

In [41]:
lat = 33.9   # system latitude (degrees)
lon = 151.2  # system longitude (degrees)

# Tell pandas how to handle nonexistent time
times = pd.DatetimeIndex(
    ausgrid_ppv["datetime"].unique()
).tz_localize(
    "Australia/Sydney",
    ambiguous="NaT",
    nonexistent="shift_forward"
)

# Remove the ambiguous timestamps
times = times.dropna()

# Calculate solar position using pvlib
solar_position = pvlib.solarposition.get_solarposition(
    time=times,
    latitude=lat,
    longitude=lon
)

# Create dataframe with datetime, zenith, and azimuth
sun = pd.DataFrame({
    "datetime": times,
    "solar_zenith": solar_position["zenith"].values,
    "solar_azimuth": solar_position["azimuth"].values})

# Remove timezone information so datetime matches ausgrid_ppv
sun["datetime"] = sun["datetime"].dt.tz_localize(None)
sun.head()

,datetime,solar_zenith,solar_azimuth
0,2010-07-01 00:00:00,122.947251,0.311255
1,2010-07-01 00:30:00,122.470484,8.487010
2,2010-07-01 01:00:00,121.126524,16.445188
3,2010-07-01 01:30:00,118.973260,24.013763
4,2010-07-01 02:00:00,116.093448,31.080145


### Plane-of-array Irradiance
Plane-of-array POA irradiance is the total solar irradiance incident on the PV panel surface.

$\text{POA} = \text{DNI} \cos (\theta_i) + \text{DHI}$

$\theta_i$ is the angle of incidence, which depends on:
- $\theta_z$ solar zenith angle
- $\gamma_s$ solar azimuth
- $\beta$ panel tilt
- $\gamma$ panel azimuth

For a fixed panel with tilt $\beta$ and azimuth $\gamma$, the angle of incidence $\theta_i$ can be calculated as:

$\cos(\theta_i) = \cos(\theta_z) \cos(\beta) + \sin(\theta_z) \sin(\beta) \cos(\gamma_s - \gamma)$

Unfortunately, panel tilt and azimuth are unknown. So, we cannot calculate POA irradiance directly. However, using the data we have, we may be able to estimate panel orientation.

#### Approximate panel orientation

At the time of maximum generation, assume $\theta_i \approx 0 \degree$, meaning the sun's rays are perpendicular to the panel. Therefore, the panel normal points directly towards the sun.
- If the panel tilt $\beta$ is measured from the horizontal: $\beta \approx \theta_z (t_\text{max})$.
- The panel azimuth is approximately $\gamma \approx \gamma_s (t_\text{max})$.

In [42]:
# Find the peak-generation datetime for each customer
peak = (
    ausgrid_ppv.loc[ausgrid_ppv.groupby("Customer")["PPV"].idxmax()]
    [["Customer", "datetime", "PPV"]])

# Add solar position at each customer's peak-generation time
peak = peak.merge(
    sun[["datetime", "solar_zenith", "solar_azimuth"]],
    on="datetime",
    how="left")

# Estimate panel orientation
panel_orientation = peak[[
    "Customer",
    "solar_zenith",
    "solar_azimuth"
]].rename(columns={
    "solar_zenith": "panel_tilt",
    "solar_azimuth": "panel_azimuth"})

panel_orientation = panel_orientation.sort_values("Customer").reset_index(drop=True)

panel_orientation.head()

,Customer,panel_tilt,panel_azimuth
0,1,52.803409,195.253396
1,2,36.374547,186.039277
2,3,42.937911,186.942349
3,4,46.657398,207.378890
4,5,42.937911,186.942349


#### Calculate angle of incidence

In [46]:
# Remove existing columns if present
ausgrid_ppv = ausgrid_ppv.drop(
    columns=["solar_zenith", "solar_azimuth", "panel_tilt", "panel_azimuth"],
    errors="ignore")

# Merge
ausgrid_ppv = ausgrid_ppv.merge(
    sun,
    on="datetime",
    how="left")

ausgrid_ppv = ausgrid_ppv.merge(
    panel_orientation,
    on="Customer",
    how="left")

ausgrid_ppv.head()

,Customer,Generator Capacity,Postcode,datetime,PPV,temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover,direct_normal_irradiance,diffuse_radiation,sunshine_duration,sun_position,solar_zenith,solar_azimuth,panel_tilt,panel_azimuth
0,1,3.78,2076,2010-07-01 00:00:00,0.0,3.60,8.30,0.0,1.0,0.0,0.0,0.0,122.979516,122.947251,0.311255,52.803409,195.253396
1,1,3.78,2076,2010-07-01 00:30:00,0.0,3.45,8.35,0.0,5.0,0.0,0.0,0.0,122.534587,122.470484,8.487010,52.803409,195.253396
2,1,3.78,2076,2010-07-01 01:00:00,0.0,3.30,8.40,0.0,9.0,0.0,0.0,0.0,121.220214,121.126524,16.445188,52.803409,195.253396
3,1,3.78,2076,2010-07-01 01:30:00,0.0,2.75,8.45,0.0,15.5,0.0,0.0,0.0,119.093277,118.973260,24.013763,52.803409,195.253396
4,1,3.78,2076,2010-07-01 02:00:00,0.0,2.20,8.50,0.0,22.0,0.0,0.0,0.0,116.236036,116.093448,31.080145,52.803409,195.253396


In [47]:
# Convert angles to radians
theta_z = np.radians(ausgrid_ppv["solar_zenith"])
gamma_s = np.radians(ausgrid_ppv["solar_azimuth"])
beta = np.radians(ausgrid_ppv["panel_tilt"])
gamma = np.radians(ausgrid_ppv["panel_azimuth"])

# Calculate cos(theta_i)
cos_theta_i = (
    np.cos(theta_z) * np.cos(beta)
    + np.sin(theta_z) * np.sin(beta)
    * np.cos(gamma_s - gamma))

# Numerical rounding can occasionally push values slightly outside [-1, 1]
cos_theta_i = np.clip(cos_theta_i, -1, 1)

# Calculate angle of incidence in degrees
ausgrid_ppv["angle_of_incidence"] = np.degrees(
    np.arccos(cos_theta_i))

ausgrid_ppv.head()

,Customer,Generator Capacity,Postcode,datetime,PPV,temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover,direct_normal_irradiance,diffuse_radiation,sunshine_duration,sun_position,solar_zenith,solar_azimuth,panel_tilt,panel_azimuth,angle_of_incidence
0,1,3.78,2076,2010-07-01 00:00:00,0.0,3.60,8.30,0.0,1.0,0.0,0.0,0.0,122.979516,122.947251,0.311255,52.803409,195.253396,167.071071
1,1,3.78,2076,2010-07-01 00:30:00,0.0,3.45,8.35,0.0,5.0,0.0,0.0,0.0,122.534587,122.470484,8.487010,52.803409,195.253396,172.711115
2,1,3.78,2076,2010-07-01 01:00:00,0.0,3.30,8.40,0.0,9.0,0.0,0.0,0.0,121.220214,121.126524,16.445188,52.803409,195.253396,173.850525
3,1,3.78,2076,2010-07-01 01:30:00,0.0,2.75,8.45,0.0,15.5,0.0,0.0,0.0,119.093277,118.973260,24.013763,52.803409,195.253396,168.988341
4,1,3.78,2076,2010-07-01 02:00:00,0.0,2.20,8.50,0.0,22.0,0.0,0.0,0.0,116.236036,116.093448,31.080145,52.803409,195.253396,162.585127


#### Compute plane-of-array irradiance

In [49]:
ausgrid_ppv["POA"] = (
    ausgrid_ppv["direct_normal_irradiance"]
    * np.maximum(
        np.cos(np.radians(ausgrid_ppv["angle_of_incidence"])), 0)
    + ausgrid_ppv["diffuse_radiation"])

,Customer,Generator Capacity,Postcode,datetime,PPV,temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover,direct_normal_irradiance,diffuse_radiation,sunshine_duration,sun_position,solar_zenith,solar_azimuth,panel_tilt,panel_azimuth,angle_of_incidence,POA
0,1,3.78,2076,2010-07-01 00:00:00,0.0,3.60,8.30,0.0,1.0,0.0,0.0,0.0,122.979516,122.947251,0.311255,52.803409,195.253396,167.071071,0.0
1,1,3.78,2076,2010-07-01 00:30:00,0.0,3.45,8.35,0.0,5.0,0.0,0.0,0.0,122.534587,122.470484,8.487010,52.803409,195.253396,172.711115,0.0
2,1,3.78,2076,2010-07-01 01:00:00,0.0,3.30,8.40,0.0,9.0,0.0,0.0,0.0,121.220214,121.126524,16.445188,52.803409,195.253396,173.850525,0.0
3,1,3.78,2076,2010-07-01 01:30:00,0.0,2.75,8.45,0.0,15.5,0.0,0.0,0.0,119.093277,118.973260,24.013763,52.803409,195.253396,168.988341,0.0
4,1,3.78,2076,2010-07-01 02:00:00,0.0,2.20,8.50,0.0,22.0,0.0,0.0,0.0,116.236036,116.093448,31.080145,52.803409,195.253396,162.585127,0.0
